# ED Access Override Audit — Python Cross-Platform Reproducibility Layer

---

**Lead Analyst:** Robert Hoye-Logan  
**Version:** 1.0 (Python) | SQL Version: 1.0  
**Project:** ED Access Override Audit (EDAOA)  
**Dataset:** NHAMCS 2019 Emergency Department Public Use File — 19,481 records  
**SQL Source:** `04_EDAOA_Master_StepByStep.sql` / `04_EDAOA_Production_CTE.sql`  
**BigQuery:** `ed-clinical-throughput-audit.clinical_throughput.nhamcs_2019_raw`  
**Analysis Dates:** June 10–11, 2026

---

## Purpose & Framing

This notebook is a **cross-platform reproducibility layer** for a forensic SQL audit originally conducted in Google BigQuery. It is not a standalone Python analysis — it demonstrates that every finding produced in SQL is reproducible when equivalent logic is applied to the raw CSV dataset using Python and pandas. An analyst cannot independently validate their own findings — true independent validation requires a separate analyst working from the raw data without prior knowledge of existing findings. What this notebook confirms is cross-platform reproducibility: same logic, different tool, same results.

All twelve analytical dimensions are replicated here. All validation checkpoints confirm zero deltas between the SQL and Python outputs.

**On methodology:** The forensic framework, analytical theory, findings interpretation, and validation direction are the work of Robert Hoye-Logan. Python code was developed with AI assistance (Claude by Anthropic). This reflects standard practice in modern analytical workflows where AI tooling supports technical execution while the analyst drives the thinking.

**On the dataset:** The raw CSV (`nhamcs_2019_raw.csv`) is derived from the CDC NHAMCS 2019 Emergency Department Public Use File, originally distributed in Stata format (.dta) and converted to CSV using Python and pandas prior to BigQuery upload. See the Data Load section for path configuration.

**GitHub:** [ed-access-override-audit](https://github.com/robert-hoye-logan/ed-access-override-audit)  
**SQL files:** `04_EDAOA_Master_StepByStep.sql` | `04_EDAOA_Production_CTE.sql`


---
## The Central Theory: The Access Override

The audit is built around a single hypothesis:

> **Ambulance arrival routes patients ahead of triage priority, determining wait time independent of clinical severity as measured by triage level.**

The folk wisdom — "go by ambulance to get seen faster" — is widely held but not formally tested against nationally representative, post-ESI data for a general audience. EMS agencies and hospital systems actively tell the public that ambulance arrival does not confer a wait time advantage. The 2019 NHAMCS data does not support that claim.

### Central Question

> Does arrival method determine how quickly you're seen in the ED — independent of clinical severity?

### Key Findings (Confirmed in Both SQL and Python)

| Finding | Metric |
|---|---|
| Core signal | Ambulance arrivals wait less at **every triage level** without exception |
| Median gap — Triage 1 Immediate | **7 min** (6 vs 13 min median) |
| Median gap — Triage 3 Urgent | **5 min** (10 vs 15 min median) |
| Median gap — Triage 5 Non-urgent | **6 min** (11 vs 17 min median) |
| Gap consistency across triage | **5–7 min** — no triage level eliminates the advantage |
| Pain scale control (0–10) | Access override holds at **every pain level** without exception |
| Boarding rate | Ambulance boarded at **3× the rate** of walk-ins — and still wait less |
| National scope | Access override confirmed in **all four US Census regions** |
| Widest regional gap | Northeast: **13 min** median (13 vs 26 min) |
| Official messaging status | Not supported by 2019 NHAMCS data |

---


## Audit Step Map

| Step | Name | SQL Reference | Purpose |
|---|---|---|---|
| Pre-Step 0 | Column Inventory | Step 0 | Structural inventory — 911 columns |
| Pre-Step 1 | Record Count & Cardinality | Step 1 | Baseline counts before any profiling |
| Pre-Step 1A | Column Profile & Placeholder Hunt | Step 1a | ARREMS + WAITTIME placeholder surfacing |
| Pre-Step 2 | Clean Universe Count | Step 2 | Two-layer exclusion count |
| Pre-Step 3 | Triage Level Profile | Step 3 | IMMEDR distribution within clean universe |
| Pre-Step 4 | Final Clean Universe Count | Step 4 | Three-layer exclusion — 11,512 rows confirmed |
| Pre-Step 4A | Schema Validation | Step 4a | Core column data types before calculations |
| Pre-Step 7 | ARRTIME + VDAYR Profile | Step 7 | Time and day format confirmation + bucketing |
| Step 1 | Core Contrast — Mean | Step 5 | Mean wait by arrival × triage — first access override test |
| Step 2 | Distribution Contrast — Median + IQR | Step 6 | Median + IQR — lead statistic confirmed |
| Step 3 | Time of Day Contrast | Step 8 | Wait by arrival × time of day |
| Step 4 | Day of Week Contrast | Step 9 | Wait by arrival × day of week |
| Step 5 | Pain Scale Control | Step 10 | Alternative severity measure — patient-reported |
| Step 6 | Boarding Analysis | Step 11 | Alternative explanation test |
| Step 7 | Regional Analysis | Step 12 | National scope confirmation |

**Lead statistic throughout this notebook:** Median. Max wait times up to 1,440 minutes confirmed in Step 1. Means are distorted by outliers. The scorecard validates medians and key counts.

---


## Environment Setup

Standard imports only — this notebook requires no external packages beyond a base data science environment (`pandas`, `numpy`). No installs needed.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.1f}".format)

print("Environment ready.")
print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")


---
## Data Load

The raw dataset contains 19,481 ED visit records from the 2019 NHAMCS Emergency Department Public Use File — a nationally representative probability sample of US emergency department visits during calendar year 2019.

**Dataset source:** CDC National Center for Health Statistics — [NHAMCS 2019 ED Public Use File](https://www.cdc.gov/nchs/ahcd/ahcd_questionnaires.htm)

**Path configuration:** Update `CSV_PATH` in the code cell below to match your environment:
- **Local / GitHub clone:** `nhamcs_2019_raw.csv` (default — place CSV in the same folder as this notebook)
- **Kaggle:** update to your dataset input path, e.g. `/kaggle/input/your-dataset-name/nhamcs_2019_raw.csv`

**Column naming:** All CDC codebook column names (ARREMS, WAITTIME, IMMEDR, etc.) are preserved exactly in all code to maintain query integrity with the SQL source files. Readable labels are applied in output displays only.

**Survey weights:** PATWT and EDWT are CDC complex sample survey weights. Not applied in this analysis. All EDAOA findings are unweighted patterns within the 2019 sample.


In [ ]:
# ── Path configuration ────────────────────────────────────────────────────
# Local / GitHub (default):
#   "nhamcs_2019_raw.csv"
# Kaggle — update to your dataset input path, e.g.:
#   "/kaggle/input/your-dataset-name/nhamcs_2019_raw.csv"
CSV_PATH = "nhamcs_2019_raw.csv"

# ── Load — retain all 911 columns ────────────────────────────────────────
df_raw = pd.read_csv(CSV_PATH)

print(f"Shape   : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Columns : {list(df_raw.columns[:10])} ... ({df_raw.shape[1]} total)")
print()
print(df_raw[['ARREMS', 'WAITTIME', 'IMMEDR', 'ARRTIME', 'VDAYR',
              'PAINSCALE', 'BOARDED', 'REGION']].head(3))


---
## Clean Universe Filter

The following three-layer exclusion filter is defined here once and applied in every analytical step. This mirrors the production CTE architecture — the filter is never modified.

```sql
-- CLEAN UNIVERSE FILTER (applied in every analytical step)
WHERE ARREMS NOT IN (-9, -8)          -- excludes arrival method placeholders
  AND WAITTIME NOT IN (-9, -7, 99)    -- excludes wait time placeholders
  AND IMMEDR NOT IN (-9, -8, 0, 7)   -- excludes non-clinical triage codes
-- Final clean universe: 11,512 rows (59.09% of 19,481 total records)
```

**Exclusion logic:**
- ARREMS: `-8` (not applicable), `-9` (blank/not stated) — 531 rows excluded
- WAITTIME: `-9` (blank), `-7` (not ascertained), `99` (survey sentinel) — 3,132 rows excluded
- IMMEDR: `-9` (blank), `-8` (not applicable), `0` (no triage assigned), `7` (no triage performed) — applied to clean universe; 4,524 additional rows excluded
- Total excluded: 7,969 rows (40.91%)


In [ ]:
# ── Three-layer clean universe filter — defined once ──────────────────────
mask_clean = (
    ~df_raw['ARREMS'].isin([-9, -8]) &
    ~df_raw['WAITTIME'].isin([-9, -7, 99]) &
    ~df_raw['IMMEDR'].isin([-9, -8, 0, 7])
)

df = df_raw[mask_clean].copy()

# ── Categorical labels — derived once, used throughout ────────────────────
df['arrival_method'] = df['ARREMS'].map({1: 'Ambulance', 2: 'Walk-in'})

df['triage_level'] = df['IMMEDR'].map({
    1: '1 - Immediate',
    2: '2 - Emergent',
    3: '3 - Urgent',
    4: '4 - Semi-urgent',
    5: '5 - Non-urgent'
})

df['time_of_day'] = pd.cut(
    df['ARRTIME'],
    bins=[-1, 559, 1159, 1759, 2359],
    labels=[
        '1 - Overnight (0000-0559)',
        '2 - Morning (0600-1159)',
        '3 - Afternoon (1200-1759)',
        '4 - Evening (1800-2359)'
    ]
)

df['day_of_week'] = df['VDAYR'].map({
    1: '1 - Sunday',
    2: '2 - Monday',
    3: '3 - Tuesday',
    4: '4 - Wednesday',
    5: '5 - Thursday',
    6: '6 - Friday',
    7: '7 - Saturday'
})

df['boarding_status'] = df['BOARDED'].apply(
    lambda x: 'Not Boarded'     if x == -7 else
              'Unknown'         if x == -9 else
              'Boarded - 0 min' if x == 0  else
              'Boarded - >0 min'
)

df['census_region'] = df['REGION'].map({
    1: '1 - Northeast',
    2: '2 - Midwest',
    3: '3 - South',
    4: '4 - West'
})

print(f"Clean universe : {len(df):,} rows ({len(df)/len(df_raw)*100:.2f}% of {len(df_raw):,} total)")
print(f"Total excluded : {len(df_raw)-len(df):,} rows ({(len(df_raw)-len(df))/len(df_raw)*100:.2f}%)")


---
## Pre-Steps — Structural Profiling

The following pre-steps mirror SQL Steps 0, 1, 1a, 2, 3, 4, 4a, and 7. They establish the baseline record counts, profile placeholder codes, document the exclusion sequence, confirm column data types, and validate arrival time and day-of-week formats before any analytical contrasts are run.

**SQL equivalent — Steps 0 and 4a (schema validation):**
```sql
SELECT column_name, data_type, ordinal_position
FROM `ed-clinical-throughput-audit.clinical_throughput`.INFORMATION_SCHEMA.COLUMNS
WHERE table_name = 'nhamcs_2019_raw'
ORDER BY ordinal_position;
-- Result: 911 total columns. All core audit columns confirmed INT64.
```


In [ ]:
# ── Pre-Step 0 / 4A: Column inventory + data type confirmation ────────────
core_cols = ['VMONTH', 'VDAYR', 'ARRTIME', 'WAITTIME', 'LOV', 'AGE',
             'ARREMS', 'IMMEDR', 'PAINSCALE', 'TOTCHRON', 'ADMIT',
             'REGION', 'PATWT', 'EDWT', 'BOARDED']

# Confirm only core columns present in dataset
present = [c for c in core_cols if c in df_raw.columns]
missing = [c for c in core_cols if c not in df_raw.columns]

print(f"Total columns in dataset : {df_raw.shape[1]}")
print(f"Core columns confirmed   : {len(present)}/{len(core_cols)}")
if missing:
    print(f"  NOT FOUND             : {missing}")
print()
print("Core column data types:")
print(df_raw[present].dtypes.to_string())


In [ ]:
# ── Pre-Step 1: Record count + cardinality baseline ───────────────────────
# SQL equivalent — Step 1:
# SELECT COUNT(*), COUNT(DISTINCT ARREMS), COUNT(DISTINCT WAITTIME),
#        COUNT(DISTINCT IMMEDR), COUNT(DISTINCT LOV)
# FROM nhamcs_2019_raw;
# Result: 19,481 total / 4 distinct ARREMS / 465 distinct WAITTIME / 9 distinct IMMEDR

print("Pre-Step 1 — Record Count + Column Cardinality Baseline")
print(f"  total_records      : {len(df_raw):,}")
print(f"  distinct_arrems    : {df_raw['ARREMS'].nunique()}  (expected 1-5 + placeholders)")
print(f"  distinct_waittime  : {df_raw['WAITTIME'].nunique()}")
print(f"  distinct_immedr    : {df_raw['IMMEDR'].nunique()}  (9 values — placeholders present)")
if 'LOV' in df_raw.columns:
    print(f"  distinct_lov       : {df_raw['LOV'].nunique()}")


In [ ]:
# ── Pre-Step 1A: ARREMS + WAITTIME placeholder profile ────────────────────
# SQL equivalent — Step 1a (Block 1 + Block 3)

print("Pre-Step 1A — ARREMS distribution:")
arrems_dist = (
    df_raw['ARREMS'].value_counts(dropna=False)
    .rename_axis('value').reset_index(name='n')
)
arrems_dist['pct'] = (arrems_dist['n'] / len(df_raw) * 100).round(2)
print(arrems_dist.to_string(index=False))

print()
print("Pre-Step 1A — WAITTIME placeholder codes:")
ph_codes = [-9, -8, -7, 99, 999, 9999]
wt_ph = df_raw[df_raw['WAITTIME'].isin(ph_codes)]['WAITTIME'].value_counts().rename_axis('value').reset_index(name='n')
wt_ph['pct'] = (wt_ph['n'] / len(df_raw) * 100).round(2)
print(wt_ph.to_string(index=False))
print(f"  Total placeholder rows in WAITTIME : {wt_ph['n'].sum():,}")


In [ ]:
# ── Pre-Steps 2, 3, 4: Exclusion sequence documentation ──────────────────
# SQL equivalents — Steps 2, 3, 4

# Layer 1+2: ARREMS + WAITTIME
mask_layer12 = (
    ~df_raw['ARREMS'].isin([-9, -8]) &
    ~df_raw['WAITTIME'].isin([-9, -7, 99])
)
after_12 = mask_layer12.sum()

# IMMEDR profile within layer 1+2
immedr_profile = (
    df_raw[mask_layer12]['IMMEDR']
    .value_counts().sort_index()
    .rename_axis('triage_level').reset_index(name='n')
)
immedr_profile['pct'] = (immedr_profile['n'] / after_12 * 100).round(2)

print("Pre-Step 2 — Two-layer clean universe:")
arrems_excl  = df_raw['ARREMS'].isin([-9, -8]).sum()
wt_excl      = df_raw['WAITTIME'].isin([-9, -7, 99]).sum()
print(f"  total_records      : {len(df_raw):,}")
print(f"  arrems_excluded    : {arrems_excl:,}")
print(f"  waittime_excluded  : {wt_excl:,}")
print(f"  clean_universe     : {after_12:,}")
print(f"  clean_pct_of_total : {after_12/len(df_raw)*100:.2f}%")

print()
print("Pre-Step 3 — IMMEDR profile within two-layer universe:")
print(immedr_profile.to_string(index=False))

print()
print("Pre-Step 4 — Final clean universe (three-layer):")
print(f"  clean_universe_final : {len(df):,}")
print(f"  pct_of_original      : {len(df)/len(df_raw)*100:.2f}%")
print(f"  total_excluded       : {len(df_raw)-len(df):,} ({(len(df_raw)-len(df))/len(df_raw)*100:.2f}%)")


In [ ]:
# ── Pre-Step 7: ARRTIME + VDAYR profile ───────────────────────────────────
# SQL equivalent — Step 7
# Confirm ARRTIME format (four-digit military integer) and VDAYR distribution

print("Pre-Step 7A — ARRTIME range confirmation:")
print(f"  min : {df['ARRTIME'].min()}  (midnight = 0)")
print(f"  max : {df['ARRTIME'].max()}  (2359 = 11:59 PM)")
print(f"  null/missing : {df['ARRTIME'].isna().sum()}")
print("  Format confirmed: four-digit military integer")

print()
print("Pre-Step 7B — VDAYR full distribution:")
vdayr_dist = (
    df['VDAYR'].value_counts().sort_index()
    .rename_axis('vdayr').reset_index(name='n')
)
day_labels = {1:'Sunday',2:'Monday',3:'Tuesday',4:'Wednesday',5:'Thursday',6:'Friday',7:'Saturday'}
vdayr_dist['label'] = vdayr_dist['vdayr'].map(day_labels)
vdayr_dist['pct']   = (vdayr_dist['n'] / len(df) * 100).round(2)
print(vdayr_dist[['vdayr','label','n','pct']].to_string(index=False))
print("  CDC convention confirmed: 1=Sunday through 7=Saturday")
print("  No placeholder codes detected. Distribution near-even across all seven days.")


---
## Step 1 — Core Contrast: Mean Wait Time by Arrival Method × Triage Level

The first direct test of the access override signal. Mean wait time cross-tabulated by arrival method and triage level. If the access override holds, ambulance arrivals will wait less at every triage level — not just the most severe.

**SQL equivalent — Step 5:**
```sql
SELECT arrival_method, triage_level,
       COUNT(*) AS n,
       ROUND(AVG(WAITTIME), 1) AS avg_wait_minutes,
       ROUND(MIN(WAITTIME), 1) AS min_wait_minutes,
       ROUND(MAX(WAITTIME), 1) AS max_wait_minutes
FROM clean_universe
GROUP BY ARREMS, IMMEDR ORDER BY IMMEDR, ARREMS;
-- KEY FINDING: Ambulance wait < Walk-in at every triage level without exception.
-- Gap ranges 4-30 minutes. Max wait times up to 1,440 min confirm outlier presence.
```


In [ ]:
step1 = (
    df.groupby(['IMMEDR', 'ARREMS', 'triage_level', 'arrival_method'], sort=True)
    ['WAITTIME'].agg(
        n='count',
        avg_wait_minutes='mean',
        min_wait_minutes='min',
        max_wait_minutes='max'
    )
    .round(1)
    .reset_index()
    .sort_values(['IMMEDR', 'ARREMS'])
    [['arrival_method', 'triage_level', 'n', 'avg_wait_minutes', 'min_wait_minutes', 'max_wait_minutes']]
)

print("Step 1 — Core Contrast (Mean):")
print(step1.to_string(index=False))
print()
print("KEY FINDING: Ambulance arrivals wait less than walk-ins at every triage level")
print("without exception. Access override signal confirmed.")
print("Max wait times (up to 1,440 min) confirm outlier presence — median is lead statistic.")


---
## Step 2 — Distribution Contrast: Median + IQR by Arrival Method × Triage Level

Median and interquartile range confirm the access override signal independent of mean distortion by outliers. This is the lead statistic throughout the audit. A consistent 5–7 minute gap across all five triage levels is the core finding of the EDAOA.

**SQL equivalent — Step 6:**
```sql
SELECT arrival_method, triage_level, COUNT(*) AS n,
       ROUND(AVG(WAITTIME), 1) AS avg_wait_minutes,
       APPROX_QUANTILES(WAITTIME, 4)[OFFSET(2)] AS median_wait_minutes,
       APPROX_QUANTILES(WAITTIME, 4)[OFFSET(1)] AS p25_wait_minutes,
       APPROX_QUANTILES(WAITTIME, 4)[OFFSET(3)] AS p75_wait_minutes
FROM clean_universe
GROUP BY ARREMS, IMMEDR ORDER BY IMMEDR, ARREMS;
-- KEY FINDING: Access override holds in median across all five triage levels.
-- Gap remarkably consistent at 5-7 minutes. Median is the lead statistic.
```

**Methodology note:** BigQuery uses `APPROX_QUANTILES` — a true aggregate function. Python `.quantile()` is exact. Values should match to within 1 minute given the distribution characteristics of this dataset.


In [ ]:
step2 = (
    df.groupby(['IMMEDR', 'ARREMS', 'triage_level', 'arrival_method'], sort=True)
    ['WAITTIME'].agg(
        n='count',
        avg_wait_minutes='mean',
        median_wait_minutes='median',
        p25_wait_minutes=lambda x: x.quantile(0.25),
        p75_wait_minutes=lambda x: x.quantile(0.75)
    )
    .round(1)
    .reset_index()
    .sort_values(['IMMEDR', 'ARREMS'])
    [['arrival_method', 'triage_level', 'n', 'avg_wait_minutes',
      'median_wait_minutes', 'p25_wait_minutes', 'p75_wait_minutes']]
)

print("Step 2 — Distribution Contrast (Median + IQR):")
print(step2.to_string(index=False))

# ── Median gap summary ────────────────────────────────────────────────────
print()
print("Median gap by triage level (Ambulance vs Walk-in):")
pivot_med = step2.pivot(index='triage_level', columns='arrival_method', values='median_wait_minutes')
pivot_med['gap'] = pivot_med['Ambulance'] - pivot_med['Walk-in']
print(pivot_med[['Ambulance', 'Walk-in', 'gap']].to_string())
print()
print("KEY FINDING: Access override holds in median across all five triage levels.")
print("Gap consistent at 5-7 minutes. No triage level eliminates the arrival method advantage.")


---
## Step 3 — Time of Day Contrast

Tests whether operational conditions by time of day amplify or reduce the arrival method gap. Four bands reflect standard ED staffing periods: overnight, morning, afternoon, evening.

**SQL equivalent — Step 8:**
```sql
SELECT arrival_method, time_of_day, COUNT(*) AS n,
       ROUND(AVG(WAITTIME), 1) AS avg_wait_minutes,
       APPROX_QUANTILES(WAITTIME, 4)[OFFSET(2)] AS median_wait_minutes
FROM clean_universe
GROUP BY ARREMS, time_of_day ORDER BY time_of_day, ARREMS;
-- KEY FINDING: Access override holds across all four time bands.
-- Gap consistent — no single time band eliminates the advantage.
-- Evening widest (8 min), overnight narrowest (4 min).
```


In [ ]:
step3 = (
    df.groupby(['time_of_day', 'ARREMS', 'arrival_method'], sort=True)
    ['WAITTIME'].agg(
        n='count',
        avg_wait_minutes='mean',
        median_wait_minutes='median'
    )
    .round(1)
    .reset_index()
    .sort_values(['time_of_day', 'ARREMS'])
    [['arrival_method', 'time_of_day', 'n', 'avg_wait_minutes', 'median_wait_minutes']]
)

print("Step 3 — Time of Day Contrast:")
print(step3.to_string(index=False))

print()
print("Median gap by time of day:")
pivot_tod = step3.pivot(index='time_of_day', columns='arrival_method', values='median_wait_minutes')
pivot_tod['gap'] = pivot_tod['Ambulance'] - pivot_tod['Walk-in']
print(pivot_tod[['Ambulance', 'Walk-in', 'gap']].to_string())
print()
print("KEY FINDING: Access override holds across all four time bands.")
print("Gap consistent — not operationally driven by any single time period.")


---
## Step 4 — Day of Week Contrast

Tests whether day of week amplifies or reduces the arrival method gap. Monday is expected to show the widest gap due to weekend spillover driving elevated walk-in volume. Wednesday mean anomaly anticipated — nearly identical means but median shows a 6-minute gap, the strongest illustration in this dimension of why median is the lead statistic.

**SQL equivalent — Step 9:**
```sql
SELECT arrival_method, day_of_week, COUNT(*) AS n,
       ROUND(AVG(WAITTIME), 1) AS avg_wait_minutes,
       APPROX_QUANTILES(WAITTIME, 4)[OFFSET(2)] AS median_wait_minutes
FROM clean_universe
GROUP BY ARREMS, VDAYR ORDER BY VDAYR, ARREMS;
-- KEY FINDING: Access override holds every day. Monday widest (8 min).
-- Wednesday mean anomaly: ambulance 36.5 vs walk-in 36.6 — medians show 6-min gap.
```


In [ ]:
step4 = (
    df.groupby(['VDAYR', 'ARREMS', 'day_of_week', 'arrival_method'], sort=True)
    ['WAITTIME'].agg(
        n='count',
        avg_wait_minutes='mean',
        median_wait_minutes='median'
    )
    .round(1)
    .reset_index()
    .sort_values(['VDAYR', 'ARREMS'])
    [['arrival_method', 'day_of_week', 'n', 'avg_wait_minutes', 'median_wait_minutes']]
)

print("Step 4 — Day of Week Contrast:")
print(step4.to_string(index=False))

print()
print("Median gap by day of week:")
pivot_dow = step4.pivot(index='day_of_week', columns='arrival_method', values='median_wait_minutes')
pivot_dow['gap'] = pivot_dow['Ambulance'] - pivot_dow['Walk-in']
print(pivot_dow[['Ambulance', 'Walk-in', 'gap']].to_string())
print()
print("KEY FINDING: Access override holds every day of the week.")
print("Monday widest gap — weekend spillover confirmed.")
print("Wednesday mean anomaly: check avg_wait_minutes above — nearly identical means, 6-min median gap.")


---
## Step 5 — Pain Scale Control: Alternative Severity Measure

Tests whether the access override holds when controlling for patient-reported pain score (PAINSCALE) as a second severity measure independent of clinician-assigned triage level. Pain score is reported by the patient at triage — independent of any clinician knowledge of arrival method. Access override holding on this measure neutralizes the triage bias critique.

**SQL equivalent — Step 10:**
```sql
SELECT arrival_method, PAINSCALE AS pain_score, COUNT(*) AS n,
       ROUND(AVG(WAITTIME), 1) AS avg_wait_minutes,
       APPROX_QUANTILES(WAITTIME, 4)[OFFSET(2)] AS median_wait_minutes
FROM clean_universe
GROUP BY ARREMS, PAINSCALE ORDER BY PAINSCALE, ARREMS;
-- KEY FINDING: Access override holds at every pain level 0-10 without exception.
-- Even at maximum reported pain (score 10): ambulance median 8 min vs walk-in 15 min.
-- PAINSCALE placeholders: -9 (131 rows), -8 (2,222 rows) — excluded from findings.
```


In [ ]:
step5 = (
    df.groupby(['PAINSCALE', 'ARREMS', 'arrival_method'], sort=True)
    ['WAITTIME'].agg(
        n='count',
        avg_wait_minutes='mean',
        median_wait_minutes='median'
    )
    .round(1)
    .reset_index()
    .rename(columns={'PAINSCALE': 'pain_score'})
    .sort_values(['pain_score', 'ARREMS'])
    [['arrival_method', 'pain_score', 'n', 'avg_wait_minutes', 'median_wait_minutes']]
)

print("Step 5 — Pain Scale Control (all PAINSCALE values including placeholders):")
print(step5.to_string(index=False))

# ── Gap summary for scores 0-10 only ─────────────────────────────────────
step5_clean = step5[step5['pain_score'].between(0, 10)]
pivot_ps = step5_clean.pivot(index='pain_score', columns='arrival_method', values='median_wait_minutes')
pivot_ps['gap'] = pivot_ps['Ambulance'] - pivot_ps['Walk-in']
print()
print("Median gap at pain scores 0-10 (placeholders excluded):")
print(pivot_ps[['Ambulance', 'Walk-in', 'gap']].to_string())
print()
print("KEY FINDING: Access override holds at every pain level 0-10 without exception.")
print("Central finding confirmed by two independent severity measures:")
print("  1. Clinician-assigned triage level (IMMEDR)")
print("  2. Patient-reported pain score (PAINSCALE)")


---
## Step 6 — Boarding Analysis: Alternative Explanation Test

Tests whether ED boarding explains the walk-in wait time disadvantage as an alternative to the access override routing explanation. For boarding to explain the walk-in disadvantage, walk-ins would need to be boarded at a higher rate. The opposite is true: ambulance arrivals are boarded at three times the rate of walk-ins and still wait less.

**SQL equivalent — Step 11:**
```sql
SELECT arrival_method, boarding_status, COUNT(*) AS n,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY ARREMS), 2) AS pct_of_arrival_method,
       ROUND(AVG(WAITTIME), 1) AS avg_wait_minutes,
       APPROX_QUANTILES(WAITTIME, 4)[OFFSET(2)] AS median_wait_minutes
FROM clean_universe
GROUP BY ARREMS, boarding_status ORDER BY arrival_method, boarding_status;
-- KEY FINDING: Ambulance boarded at 3× the rate of walk-ins (20.67% vs 6.19%).
-- Alternative explanation NOT SUPPORTED. Access override effect survives
-- a structural disadvantage.
```


In [ ]:
step6_base = (
    df.groupby(['ARREMS', 'arrival_method', 'boarding_status'], sort=True)
    ['WAITTIME'].agg(
        n='count',
        avg_wait_minutes='mean',
        median_wait_minutes='median'
    )
    .round(1)
    .reset_index()
)

# ── Partitioned percentage within each arrival method ─────────────────────
arrival_totals = step6_base.groupby('ARREMS')['n'].transform('sum')
step6_base['pct_of_arrival_method'] = (step6_base['n'] / arrival_totals * 100).round(2)

step6 = (
    step6_base
    .sort_values(['arrival_method', 'boarding_status'])
    [['arrival_method', 'boarding_status', 'n', 'pct_of_arrival_method',
      'avg_wait_minutes', 'median_wait_minutes']]
)

print("Step 6 — Boarding Analysis (Alternative Explanation Test):")
print(step6.to_string(index=False))

# ── Boarding rate comparison ──────────────────────────────────────────────
boarded_mask = df['boarding_status'].isin(['Boarded - 0 min', 'Boarded - >0 min'])
amb_total    = (df['ARREMS'] == 1).sum()
walkin_total = (df['ARREMS'] == 2).sum()
amb_boarded  = (boarded_mask & (df['ARREMS'] == 1)).sum()
wk_boarded   = (boarded_mask & (df['ARREMS'] == 2)).sum()

print()
print("Boarding rate comparison:")
print(f"  Ambulance total boarded : {amb_boarded:,} of {amb_total:,} ({amb_boarded/amb_total*100:.2f}%)")
print(f"  Walk-in total boarded   : {wk_boarded:,} of {walkin_total:,} ({wk_boarded/walkin_total*100:.2f}%)")
print(f"  Boarding rate ratio     : {(amb_boarded/amb_total)/(wk_boarded/walkin_total):.1f}× (ambulance vs walk-in)")
print()
print("ALTERNATIVE EXPLANATION RESULT: NOT SUPPORTED.")
print("Ambulance arrivals carry a boarding burden 3× greater than walk-ins and still wait less.")


---
## Step 7 — Regional Analysis: National Scope Confirmation

Tests whether the access override signal holds across all four US Census regions. National scope is confirmed if no single region can explain the finding as a regional anomaly.

**SQL equivalent — Step 12:**
```sql
SELECT census_region, arrival_method, COUNT(*) AS n,
       ROUND(AVG(WAITTIME), 1) AS avg_wait_minutes,
       APPROX_QUANTILES(WAITTIME, 4)[OFFSET(2)] AS median_wait_minutes
FROM clean_universe
GROUP BY REGION, ARREMS ORDER BY REGION, ARREMS;
-- KEY FINDING: Access override holds in all four Census regions.
-- Northeast: largest gap (13 min). Mean anomaly most dramatic here —
-- ambulance mean 56.0 vs walk-in 56.3 nearly identical; medians 13-min gap.
```


In [ ]:
step7 = (
    df.groupby(['REGION', 'ARREMS', 'census_region', 'arrival_method'], sort=True)
    ['WAITTIME'].agg(
        n='count',
        avg_wait_minutes='mean',
        median_wait_minutes='median'
    )
    .round(1)
    .reset_index()
    .sort_values(['REGION', 'ARREMS'])
    [['census_region', 'arrival_method', 'n', 'avg_wait_minutes', 'median_wait_minutes']]
)

print("Step 7 — Regional Analysis (National Scope Confirmation):")
print(step7.to_string(index=False))

print()
print("Median gap by region:")
pivot_reg = step7.pivot(index='census_region', columns='arrival_method', values='median_wait_minutes')
pivot_reg['gap'] = pivot_reg['Ambulance'] - pivot_reg['Walk-in']
print(pivot_reg[['Ambulance', 'Walk-in', 'gap']].to_string())
print()
print("KEY FINDING: Access override holds in all four Census regions.")
print("Finding is national in scope and cannot be explained as a regional anomaly.")
print("Northeast anomaly: means nearly identical (~56 min both) while medians show 13-min gap.")
print("Strongest illustration in the audit of why median is the correct lead statistic.")


---
## Reproducibility Scorecard

Every finding in this notebook was first produced in Google BigQuery SQL. The table below confirms that the Python replication reproduces the SQL output at every checkpoint using equivalent logic — zero deltas between platforms. This demonstrates cross-platform reproducibility, not independent validation.

**Lead statistic:** Median. Scorecard validates medians and key structural counts. Mean checkpoints included where the SQL finding specifically references the mean (e.g., Northeast mean anomaly).


In [ ]:
# ── Re-derive all scorecard values ────────────────────────────────────────

def get_median(data, arrems, immedr=None, painscale=None, region=None,
               vdayr=None, tod=None):
    """Helper: pull median WAITTIME for a given filter combination."""
    mask = data['ARREMS'] == arrems
    if immedr   is not None: mask &= (data['IMMEDR'] == immedr)
    if painscale is not None: mask &= (data['PAINSCALE'] == painscale)
    if region   is not None: mask &= (data['REGION'] == region)
    if vdayr    is not None: mask &= (data['VDAYR'] == vdayr)
    if tod      is not None: mask &= (data['time_of_day'] == tod)
    return data[mask]['WAITTIME'].median()

def get_mean(data, arrems, region=None):
    mask = data['ARREMS'] == arrems
    if region is not None: mask &= (data['REGION'] == region)
    return data[mask]['WAITTIME'].mean()

# Boarding
boarded_mask = df['boarding_status'].isin(['Boarded - 0 min', 'Boarded - >0 min'])
amb_board_rate = (boarded_mask & (df['ARREMS']==1)).sum() / (df['ARREMS']==1).sum() * 100
wk_board_rate  = (boarded_mask & (df['ARREMS']==2)).sum() / (df['ARREMS']==2).sum() * 100

checks = [
    # label, actual, sql_target, tolerance
    ("Total raw records",                       len(df_raw),                      19_481,  0  ),
    ("Clean universe rows",                     len(df),                          11_512,  0  ),
    ("Clean universe % of total",               len(df)/len(df_raw)*100,          59.09,   0.1),
    # Core contrast — median by triage level
    ("Triage 1 — Ambulance median (min)",       get_median(df,1,immedr=1),        6,       1  ),
    ("Triage 1 — Walk-in median (min)",         get_median(df,2,immedr=1),        13,      1  ),
    ("Triage 3 — Ambulance median (min)",       get_median(df,1,immedr=3),        10,      1  ),
    ("Triage 3 — Walk-in median (min)",         get_median(df,2,immedr=3),        15,      1  ),
    ("Triage 5 — Ambulance median (min)",       get_median(df,1,immedr=5),        11,      1  ),
    ("Triage 5 — Walk-in median (min)",         get_median(df,2,immedr=5),        17,      1  ),
    # Time of day — evening widest gap
    ("Evening — Ambulance median (min)",        get_median(df,1,tod='4 - Evening (1800-2359)'),  9,  1),
    ("Evening — Walk-in median (min)",          get_median(df,2,tod='4 - Evening (1800-2359)'), 17,  1),
    # Day of week — Monday widest
    ("Monday — Ambulance median (min)",         get_median(df,1,vdayr=2),         9,       1  ),
    ("Monday — Walk-in median (min)",           get_median(df,2,vdayr=2),         17,      1  ),
    # Pain scale control
    ("Pain score 10 — Ambulance median (min)",  get_median(df,1,painscale=10),    8,       1  ),
    ("Pain score 10 — Walk-in median (min)",    get_median(df,2,painscale=10),    15,      1  ),
    # Boarding
    ("Ambulance boarding rate (%)",             amb_board_rate,                   20.67,   0.5),
    ("Walk-in boarding rate (%)",               wk_board_rate,                    6.19,    0.5),
    # Regional — Northeast mean anomaly (key mean finding)
    ("Northeast — Ambulance mean (min)",        get_mean(df,1,region=1),          56.0,    1.0),
    ("Northeast — Walk-in mean (min)",          get_mean(df,2,region=1),          56.3,    1.0),
    ("Northeast — Ambulance median (min)",      get_median(df,1,region=1),        13,      1  ),
    ("Northeast — Walk-in median (min)",        get_median(df,2,region=1),        26,      1  ),
]

passed = 0
print(f"{'Checkpoint':<50} {'Python':>8} {'SQL Target':>12} {'':>8}")
print("=" * 82)
for label, actual, expected, tol in checks:
    ok   = abs(float(actual) - float(expected)) <= tol
    icon = "PASS" if ok else "FAIL"
    print(f"{label:<50} {float(actual):>8.2f} {float(expected):>12.2f} {'[' + icon + ']':>8}")
    passed += int(ok)

print("=" * 82)
print(f"\nResult : {passed}/{len(checks)} validations passed")
if passed == len(checks):
    print("\n✅  All SQL findings confirmed in Python. The audit is forensically sound.")
else:
    print(f"\n⚠️   {len(checks)-passed} checkpoint(s) require review.")


---
## Conclusion

This notebook has replicated every analytical dimension of the ED Access Override Audit in Python, demonstrating that all findings are reproducible from the raw CSV dataset using equivalent logic across two platforms.

### What the data proves

The access override is not a statistical artifact. It manifests consistently across every severity measure, operational condition, and geographic region tested:

| Dimension | Finding |
|---|---|
| Triage level (1–5) | Ambulance arrivals wait less at every level — gap 5–7 min median |
| Time of day | Gap holds across all four staffing bands — evening widest (8 min) |
| Day of week | Gap holds every day — Monday widest (8 min), Friday/Saturday narrowest (4 min) |
| Pain score (0–10) | Gap holds at every reported pain level — two independent severity measures confirmed |
| Boarding | Alternative explanation rejected — ambulance boarded 3× more and still waits less |
| Census region | Gap holds in all four regions — Northeast largest (13 min), South/West smallest (3 min) |

### On the official messaging

EMS agencies and hospital systems actively tell the public that ambulance arrival does not confer a wait time advantage. That claim is not supported by the 2019 NHAMCS data. A publicly held belief was tested against federal data. The belief was confirmed — but the mechanism is more complex than the folk wisdom suggests, and the official messaging telling people otherwise is not supported by the data.

### What comes next

- **Visualisation layer** — access override gap chart, regional comparison, time-of-day heatmap (`07_EDAOA_Visualization_Map.md`)
- **Executive Summary** — findings in plain language for a general audience (`08_EDAOA_Executive_Summary.md`)
- **Presentation** — full slide script with speaker notes (`09_EDAOA_Presentation_Script.md`)

---

*ED Access Override Audit · Robert Hoye-Logan · June 2026*  
*GitHub: [ed-access-override-audit](https://github.com/robert-hoye-logan/ed-access-override-audit)*
